# RisingBALLER MPP — Local Test Run

Small model for quick local validation that the full pipeline works.
- Tiny embed_size (32), 1 transformer layer, 2 heads
- Pre-collated batches (batch=32, repeat=2) — same pipeline as main training
- Few epochs
- Verifies: dataset → collator → pre-collation → model → trainer → metrics

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "RisingBaller":
    ROOT = ROOT.parent.parent
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)
print("ROOT:", ROOT)

## 1. Data & Vocab

In [ ]:
from data.preprocessing import preprocess_raw_csv, build_vocab_mappings

raw_path = ROOT / "dataset" / "data_with_dates.csv"
sample_path = ROOT / "notebooks" / "train_sample_raw.csv"
output_dir = str(ROOT / "notebooks" / "train_sample_processed")

df_raw = pd.read_csv(raw_path)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print(f"Matches: {df['match_id'].nunique()}")
print(f"players_vocab_size (pad_idx): {vocab['player_pad_token_id']}")
print(f"MPP classes (real players): {vocab['player_pad_token_id'] - 1}")

## 2. Dataset & Pre-collated Batches

In [ ]:
from torch.utils.data import DataLoader
from data.dataset import MatchDatasetMPP, PreCollatedDataset
from data.collator import DataCollatorMPP
try:
    from data.collator import DataCollatorPreCollated
except ImportError:
    from torch.utils.data.dataloader import default_collate
    class DataCollatorPreCollated:
        def __call__(self, batch):
            return default_collate(batch)

sample_batch_size = 32
repeat = 2
dev_ratio = 0.05
seed = 42
max_seq_length = 36

# ---- Split by match_id BEFORE creating batches (no data leakage) ----
match_ids = df["match_id"].unique()
np.random.seed(seed)
np.random.shuffle(match_ids)
dev_size = max(1, int(len(match_ids) * dev_ratio))
dev_match_ids = set(match_ids[:dev_size])
train_match_ids = set(match_ids[dev_size:])

df_train = df[df["match_id"].isin(train_match_ids)].reset_index(drop=True)
df_dev = df[df["match_id"].isin(dev_match_ids)].reset_index(drop=True)

print(f"Matches total: {len(match_ids)}, train: {len(train_match_ids)}, dev: {len(dev_match_ids)}")

# ---- Create separate datasets for train and dev ----
ds_train = MatchDatasetMPP(
    df_train,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)
ds_dev = MatchDatasetMPP(
    df_dev,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

# Sanity check
sample = ds_train[0]
print(f"Seq length: {sample['input_ids'].shape[0]}")
print(f"Occupied slots: {sample['attention_mask'].sum().item()}")

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

def _collate_filter_none(batch):
    batch = [b for b in batch if b is not None]
    return collator(batch) if batch else None

# ---- Build train batches ----
loader_train = DataLoader(
    ds_train, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=True,
)
train_batches = []
for _ in range(repeat):
    for batch in loader_train:
        if batch is not None:
            train_batches.append(batch)

# ---- Build dev batches (drop_last=False to keep all dev matches) ----
loader_dev = DataLoader(
    ds_dev, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=False,
)
dev_batches = []
for _ in range(repeat):
    for batch in loader_dev:
        if batch is not None:
            dev_batches.append(batch)

train_dataset = PreCollatedDataset(train_batches)
eval_dataset = PreCollatedDataset(dev_batches)
collator_for_trainer = DataCollatorPreCollated()

print(f"Train batches: {len(train_batches)}, Dev batches: {len(dev_batches)}")

## 3. Tiny Transformer Model & Trainer

In [ ]:
from models.transformer.pretrain import MaskedPlayerModel
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp
from training.callbacks import BestNCheckpointCallback

# Tiny model for local testing
model = MaskedPlayerModel(
    embed_size=32,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
enc_params = sum(p.numel() for p in model.encoder.parameters())
print(f"Total params: {total_params:,}")
print(f"Encoder params: {enc_params:,}")

output_dir = str(ROOT / "notebooks" / "RisingBaller" / "test_output")
training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 20,
    "eval_strategy": "steps",
    "eval_steps": 20,
    "save_strategy": "no",
    "report_to": "none",
    "seed": seed,
}

best_ckpt_cb = BestNCheckpointCallback(
    metrics={"eval_loss": "min", "eval_accuracy_top3": "max"},
    top_n=2,
)

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator_for_trainer,
)
trainer.add_callback(best_ckpt_cb)
best_ckpt_cb.set_trainer(trainer)
print("Trainer ready.")

## 4. Train

In [ ]:
trainer.train()
print("\n" + best_ckpt_cb.summary())

## 5. Evaluate

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
results = trainer.evaluate()
print("Eval results:")
for k, v in results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)
train_logs = history[history["loss"].notna()]
eval_logs = history[history["eval_loss"].notna()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], label="train")
if len(eval_logs) > 0:
    axes[0].plot(eval_logs["step"], eval_logs["eval_loss"], label="eval")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

if "eval_accuracy_top1" in eval_logs.columns:
    axes[1].plot(eval_logs["step"], eval_logs["eval_accuracy_top1"], label="top1")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Top-1 Accuracy")
    axes[1].legend()

if "eval_accuracy_top3" in eval_logs.columns:
    axes[2].plot(eval_logs["step"], eval_logs["eval_accuracy_top3"], label="top3")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Accuracy")
    axes[2].set_title("Top-3 Accuracy")
    axes[2].legend()

plt.tight_layout()
plt.show()